# Evaluating 2f Cuts

In [ ]:
import sqlite3
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

DB    = '/data/users/class/dpipe_db/w1_templates_20260203.db'
NDET  = 616
MINMED = 0.005
MAXVAR = 1e-6
MINBASE = 0.0125
QTHR = 0.95

In [2]:
# load in template table from database
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)

pf = pd.read_sql_query(
    "SELECT det_uid AS uid, "
    "segment_id, "
    "ABS(amplitude*scaling_factor) AS a, "
    "quality "
    "FROM templates",
    con
)

con.close()

# load in segment/span table from database
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)

df_spans = pd.read_sql_query(
    "SELECT *"
    "FROM segments",
    con
)

con.close()

In [9]:
full_pf = pd.merge(pf, df_spans, left_on='segment_id', right_on='id', how='inner').drop(columns=['id', 'subdivision_index', 'mod_freq', 'modulator_type', 'site_pwv'])


In [ ]:
NSPAN = full_pf['span_id'].max()

# get good quality data - we only calculate span median, variance, IQR bounds, etc. from this
good_pf = full_pf[full_pf["quality"] >= 0.5]
#g = full_pf.groupby(['uid', 'span_id'])
g = good_pf.groupby(['uid', 'span_id'])

uids = np.sort(full_pf['uid'].unique())
span_ids = np.sort(full_pf['span_id'].unique())

# get mean, variance, number of datapkg's for each detector, span pair
pmean = np.full(NDET, np.nan); pmed = np.full((NDET, NSPAN+1), np.nan)
pvar = np.full((NDET, NSPAN+1), np.nan); pn = np.zeros((NDET, NSPAN+1))
for (uid, span_id), sub in g:
    vals = sub["a"].values
    pmed[uid, span_id] = np.median(vals)
    pvar[uid, span_id] = np.var(vals)
    pn[uid, span_id]   = len(vals)

# SPAN-WIDE CUTS: ModTemplateLowMedianCut, ModTemplateHighVarianceCut
# only keep nonempty spans with median 2f amp >= MINMED, variance <= MAXVAR
span_survive = (pmed >= MINMED) & (pvar <= MAXVAR) & (pn > 0)
det_keep, span_keep = np.where(span_survive)

surviving_pairs = set(zip(det_keep, span_keep))
surviving_pairs_df = pd.DataFrame(
    surviving_pairs,
    columns=["uid", "span_id"]
)

pf_surviving_spans = full_pf.merge(
    surviving_pairs_df,
    on=["uid", "span_id"],
    how="inner"
)

# define IQR bounds of each detector
amp_stats = good_pf.groupby("uid")["a"].agg(
    q1=lambda x: x.quantile(0.25),
    q3=lambda x: x.quantile(0.75)
)

lower_bound = np.full(NDET, np.nan)
upper_bound = np.full(NDET, np.nan)

for uid, row in amp_stats.iterrows():
    if uid < NDET:
        q1 = row["q1"]
        q3 = row["q3"]
        iqr = q3 - q1

        lower_bound[uid] = q1 - 1.5 * iqr
        upper_bound[uid] = q3 + 1.5 * iqr


print("done with database prep")

done with database prep


In [ ]:
# DATAPKG CUTS: ModTemplateQualityAmpConditionCut and ModTemplateAmplitudeOutlierCut
pf_datapkg_cut = pf_surviving_spans[
    (pf_surviving_spans["quality"] >= QTHR) &
    (pf_surviving_spans["a"] >= MINBASE) &
    (pf_surviving_spans["a"] >= lower_bound[pf_surviving_spans["uid"]]) &
    (pf_surviving_spans["a"] <= upper_bound[pf_surviving_spans["uid"]])
]

pf_datapkg_cut

,uid,segment_id,a,quality,span_id,timestamp
0,0,1,0.015997,0.999999,1,1.723702e+09
1,0,2,0.015925,0.999999,1,1.723702e+09
2,0,3,0.015991,0.999999,1,1.723702e+09
3,0,4,0.015995,0.999999,1,1.723703e+09
4,0,5,0.015993,0.999999,1,1.723703e+09
...,...,...,...,...,...,...
22843387,361,86602,0.021197,0.999996,305,1.756379e+09
22843388,361,86603,0.021126,0.999996,305,1.756379e+09
22843389,361,86604,0.021130,0.999996,305,1.756380e+09
22843390,361,86605,0.021113,0.999996,305,1.756380e+09
